# DS 227 · Week 8 — Data exploration II
## Relationships between variables

Every code block from the lecture, in order, so you can run what you watched instead of
retyping it. The first slides use small tables typed into the cells; the **Case study**
cell loads the flood control data from the course API, so there is no file to download.

**How to use this.** Run the cells top to bottom the first time (Runtime → Run all). The
exercise cells list their tasks above the code: predict first, then run, then change one
thing and run again.

**Version differences.** The slides follow pandas 3. On pandas 2, text columns show as
`object` instead of `str`, and dates as `datetime64[ns]` instead of `datetime64[us]`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---
## The lecture, cell by cell

Headings carry the slide numbers shown in the deck's counter.


### Slide 5 — A trend is hard to see in a table


Expected output:

```
study_hours  sleep_hours  score
0            2            8     65
1            5            6     82
2            1            9     60
3            8            5     95
4            6            6     88
5            3            8     70
6            7            5     90
7            4            7     78
```


In [ ]:
students = pd.DataFrame({
    "study_hours": [2, 5, 1, 8, 6, 3, 7, 4],
    "sleep_hours": [8, 6, 9, 5, 6, 8, 5, 7],
    "score":       [65, 82, 60, 95, 88, 70, 90, 78],
})

print(students)

### Slide 8 — Code and charts on these slides run in the browser


Expected output:

```
pandas version: 3.0.5
64-bit Python: True
city      str
pop     int64
dtype: object
```


In [ ]:
import io, sys

print("pandas version:", pd.__version__)
print("64-bit Python:", sys.maxsize > 2**32)

t = pd.read_csv(io.StringIO("city,pop\nCebu,964000"))
print(t.dtypes)

### Slide 11 — .corr() builds the whole matrix


**Exercise.**

1. Predict the sign of the sleep_hours–score correlation.
2. Run the cell.
3. Change the third student’s sleep from 9 to 4 hours and run again. What happens to that correlation?


In [ ]:
# students: the eight students from the first slide
print(students.corr().round(2))

### Slide 12 — The flood data, loaded from the course API


Expected output:

```
(34079, 16)
```


In [ ]:
import io, requests

url  = "https://portal.latarak.com/api/flood/export.csv"
resp = requests.get(url, timeout=60)
df   = pd.read_csv(io.StringIO(resp.text))

# week 6's derived column: days from start to completion
for col in ["start_date", "completion_date"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")
df["duration"] = (df.completion_date - df.start_date).dt.days

cols = ["budget", "duration", "progress", "year", "latitude", "longitude"]
print(df.shape)

### Slide 13 — Six columns, every pair at once


Expected output:

```
budget    dur   prog   year    lat    lon
budget   1.000  0.365 -0.021  0.265 -0.054  0.047
dur      0.365  1.000 -0.096 -0.057 -0.104  0.078
prog    -0.021 -0.096  1.000 -0.379 -0.026  0.049
year     0.265 -0.057 -0.379  1.000  0.013 -0.027
lat     -0.054 -0.104 -0.026  0.013  1.000 -0.785
lon      0.047  0.078  0.049 -0.027 -0.785  1.000
```


In [ ]:
short = df[cols].rename(columns={"duration": "dur", "progress": "prog",
                                 "latitude": "lat", "longitude": "lon"})
print(short.corr().round(3))

### Slide 14 — Larger projects take moderately longer


Expected output:

```
27664
0.365
0.133
```


In [ ]:
sub = df[["budget", "duration"]].dropna()
print(len(sub))

print(round(sub.budget.corr(sub.duration), 3))
print(round(0.365**2, 3))     # r squared

### Slide 15 — Two coefficients for the same pair; the gap is informative


Expected output:

```
0.365
```


In [ ]:
# sub: budget and duration, both present
# Pearson: straight-line strength
print(round(sub.budget.corr(sub.duration), 3))

### Slide 15 — Two coefficients for the same pair; the gap is informative


Expected output:

```
0.508
```


In [ ]:
# Spearman: agreement of the RANKS
print(round(sub.budget.rank().corr(sub.duration.rank()), 3))

### Slide 16 — Logging the skewed variable raises Pearson


Expected output:

```
0.396
```


In [ ]:
lg = sub[sub.budget > 0]
print(round(np.log10(lg.budget).corr(lg.duration), 3))

### Slide 17 — r = 0.365 examined by budget quartile


Expected output:

```
band
Q1 small    119.0
Q2          189.0
Q3          226.0
Q4 large    269.0
Name: duration, dtype: float64
```


In [ ]:
sub["band"] = pd.qcut(sub.budget, 4,
    labels=["Q1 small", "Q2", "Q3", "Q4 large"])

print(sub.groupby("band", observed=True).duration.median())

### Slide 21 — The correlation reflects the shape of the country


Expected output:

```
30884
-0.785
[5.01, 20.79]
[117.06, 126.57]
```


In [ ]:
ll = df[["latitude", "longitude"]].dropna()
print(len(ll))

print(round(ll.latitude.corr(ll.longitude), 3))
print(ll.latitude.agg(["min", "max"]).round(2).tolist())
print(ll.longitude.agg(["min", "max"]).round(2).tolist())

### Slide 26 — The scatter plot: one dot per row


**Exercise.**

1. Predict the shape the eight dots will make.
2. Run the cell.
3. Change "study_hours" to "sleep_hours" in the scatter line. What changes?


In [ ]:
# students: the eight students from the first slide
fig, ax = plt.subplots()
ax.scatter(students["study_hours"], students["score"])
ax.set_xlabel("Study hours")
ax.set_ylabel("Score")
plt.show()

### Slide 28 — One straight line through the dots


**Exercise.**

1. Predict roughly how many points each extra hour of study adds.
2. Run the cell.
3. Change xs to np.array([1, 10]). What score does the line predict at 10 hours?


In [ ]:
# students: the eight students from the first slide
import numpy as np

slope, intercept = np.polyfit(students["study_hours"],
                              students["score"], 1)
print(round(slope, 1), round(intercept, 1))

xs = np.array([1, 8])
fig, ax = plt.subplots()
ax.scatter(students["study_hours"], students["score"])
ax.plot(xs, slope * xs + intercept)
plt.show()

### Slide 30 — Overplotting in a large scatter plot


In [ ]:
df.plot.scatter(x="budget", y="duration")
plt.show()

# fix: a reproducible sample of 2,000 projects
df.sample(2000, random_state=42).plot.scatter(x="budget", y="duration")
plt.show()

# fix: count the projects in hexagonal cells
df.plot.hexbin(x="budget", y="duration", gridsize=40)
plt.show()

### Slide 33 — The same plot on a log axis


In [ ]:
ax = df.plot.scatter(x="budget", y="duration",
                     alpha=0.05)
ax.set_xscale("log")
plt.show()

### Slide 35 — Heatmap: the matrix in colour


In [ ]:
# students: the eight students from the first slide
import seaborn as sns

sns.heatmap(students.corr(), annot=True,
            cmap="coolwarm", center=0)
plt.show()

### Slide 37 — The pair plot: the matrix as scatter plots


Expected output:

```
(3, 3)
```


In [ ]:
# students: the eight students from the first slide
import seaborn as sns

g = sns.pairplot(students)
print(g.axes.shape)
plt.show()

# add hue= to colour every panel by a group,
# e.g. sns.pairplot(students, hue="passed")

### Slide 39 — The correlation matrix as a heatmap


In [ ]:
import seaborn as sns

sns.heatmap(df[cols].corr(),
            annot=True, fmt=".2f",
            cmap="RdBu_r", center=0,
            vmin=-1, vmax=1)
plt.show()

### Slide 42 — Coloring the dots by a category


In [ ]:
# students: the eight students from the first slide
import seaborn as sns

students["passed"] = students["score"] >= 75
sns.scatterplot(data=students,
    x="study_hours", y="score",
    hue="passed")
plt.show()

### Slide 43 — Colour, size or a separate panel


In [ ]:
import seaborn as sns

# keep every column, drop rows missing either number
d = df.dropna(subset=["budget", "duration"])

# a category becomes colour
sns.scatterplot(data=d, x="budget", y="duration",
                hue="status", alpha=0.3)
plt.show()

### Slide 44 — Eighteen regions, one shared scale


In [ ]:
import seaborn as sns

# d: the projects with both numbers (previous slide)
g = sns.FacetGrid(d, col="region", col_wrap=4)
g.map(plt.scatter, "budget", "duration")
plt.show()

### Slide 45 — Figure, axes, subplots and a shared axis


Expected output:

```
(2,)
shared y: True
```


In [ ]:
# students: the eight students from the first slide
fig, axes = plt.subplots(1, 2, figsize=(8, 3),
                         sharey=True)
axes[0].scatter(students["study_hours"], students["score"])
axes[1].scatter(students["sleep_hours"], students["score"])

print(axes.shape)
print("shared y:", axes[0].get_ylim() == axes[1].get_ylim())
plt.show()

### Slide 50 — Split, then summarize


**Exercise.**

1. Predict which group studied more on average.
2. Run the cell.
3. Change 75 to 85. How do the two groups change?


In [ ]:
# students: the eight students from the first slide
students["passed"] = students["score"] >= 75

print(students.groupby("passed")[
    ["study_hours", "sleep_hours"]
].mean().round(1))

### Slide 51 — One box per region, eighteen distributions at once


In [ ]:
import seaborn as sns

order = (df.groupby("region").duration
           .median().sort_values().index)

sns.boxplot(data=df, x="duration", y="region",
            order=order)
plt.show()

### Slide 54 — Rates, not counts, when the groups differ in size


Expected output:

```
region
Region IX     0.656
Region XII    0.920
Name: Completed, dtype: float64
```


In [ ]:
rate = (pd.crosstab(df.region, df.status,
                    normalize="index")["Completed"]
        .sort_values().round(3))
print(rate.iloc[[0, -1]])       # lowest and highest of 18

### Slide 55 — Grouped bars compare; stacked bars add up


Expected output:

```
status  Completed  On-Going
year
2023         3842       613
2024         3216      1855
2025          375      2615
```


In [ ]:
ct = pd.crosstab(df.year, df.status)
few = ct.loc[2023:2025, ["Completed", "On-Going"]]
print(few)

few.plot.bar()               # grouped
plt.show()
few.plot.bar(stacked=True)   # stacked
plt.show()

### Slide 59 — progress vs year, r = −0.379


Expected output:

```
year
2021    99.4
2022    98.9
2023    97.2
2024    83.8
2025    57.1
2026     0.0
Name: progress, dtype: float64
```


In [ ]:
print(df.groupby("year").progress.mean().round(1).loc[2021:])

### Slide 61 — Checking whether a relationship holds within groups


Expected output:

```
region
Central Office             0.655
Region VIII                0.623
Region IV-B                0.112
National Capital Region    0.090
dtype: float64
```


In [ ]:
r = (df.dropna(subset=["budget", "duration"])
   .groupby("region")[["budget", "duration"]]
   .apply(lambda g: g.budget.corr(g.duration))
   .sort_values(ascending=False).round(3))
print(r.iloc[[0, 1, -2, -1]])    # top two, bottom two

---

*Data: DPWH flood control projects 2016–2026, mirrored by BetterGov.ph from the DPWH
transparency portal, CC0-1.0 (public domain). Served by the course API at* `/api/flood`.
